Notebook 4

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.vision import models, transforms
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.models import ResNet18_Weights


from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
if torch.cuda.is_available():
    device_name = "cuda"
elif torch.backends.mps.is_available():
    device_name = "mps"
else:
    device_name = "cpu"
    
print(f'Using device: {device_name}')
device = torch.device(device_name)

Build Model

In [ ]:
CKPT_PATH = "models/resnet_pattern_classifier.pth"
ckpt = torch.load(CKPT_PATH, map_location=device_name)
id2label = ckpt["id2label"]
label2id = ckpt["label2id"]
num_classes = len(id2label)

model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.load_state_disct(ckpt["state_dict"])
model = model.to(DEVICE).eval()

print("Loaded model with classees:", id2label)


Load Test Data

In [ ]:
labels_df = pd.read_csv("../data/labels.csv")

image_paths = []
true_labels = []

for idx, row in labels_df.iterrows():
    folder = f"../data/frames{row['division']}/{row['division']}_{row['id']}"
    for img_path in Path(folder).glob("*.jpg"):
        image_paths.append(str(img_path))
        true_labels.append(label2id[row['pattern']])

Transform

In [ ]:
tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Loaded", len(image_paths), "frames")

Making Predictions

In [ ]:
preds = []

with torch.no_grad():
    for path in image_paths:
        img = Image.open(path).convert("RGB")
        x = tfm(img).unsqueeze(0).to(device_name)
        logits = model(x)
        pred = logits.argmax(dim=1).item()
        preds.append(pred)
        

Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=list(id2label.values()))
plt.figure(figsize=(6,5))
disp.plot(cmap='Blues', values_format="d")
plt.title("Confustion Matric - Pattern Classification for CNN")
plt.show()

Misclassified Examples

In [ ]:
mis_idx = [i for i in range(len(preds)) if preds[i] != true_labels[i]]

print("number of misclassified examples:", len(mis_idx))

def show_misclassified(n=6):
    plt.figure(figsize=(15, 5))
    for i in mis_idx[:n]:
        img = Image.open(image_paths[i]).convert("RGB")
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"True: {id2label[true_labels[i]]}, Pred: {id2label[preds[i]]}")
        plt.show()
       

Grad-CAM Implementation

In [ ]:
gradients = None
activations = None
def save_gradients(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0]

def save_activations(module, input, output):
    global activations
    activations = output

target_layer = model.layer4[-1].conv2
target_layer.register_forward_hook(save_activations)
target_layer.register_backward_hook(save_gradients)
